<a href="https://colab.research.google.com/github/Mihailby/Taxes/blob/main/notebooks/01/tut_1_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1.2 The two-variable linear program in AMPL

In [2]:
%pip install PyPDF2 reportlab pandas
import PyPDF2
from PyPDF2 import PdfReader, PdfWriter
import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 39.1 MB/s eta 0:00:00


In [6]:
# @markdown
Name = 'Mikhail Riabtsev' # @param {type:"string"}
SSN = '123-45-6789' # @param {type:"string"}
import re
def split_ssn(ssn):
    # Check if SSN is in correct format
    ssn_pattern = r'^\d{3}-\d{2}-\d{4}$'

    if not re.match(ssn_pattern, ssn):
        print("❌ Invalid SSN format. Please use XXX-XX-XXXX")
        return None, None, None

    # Split the SSN
    parts = ssn.split('-')
    SSN1 = parts[0]  # First 3 digits
    SSN2 = parts[1]  # Middle 2 digits
    SSN3 = parts[2]  # Last 4 digits

    return SSN1, SSN2, SSN3

# Split the SSN
ssn1, ssn2, ssn3 = split_ssn(SSN)

# @markdown
Street_Address = "dd"  # @param {type:"string"}
City_Town_County_ZIP_Code = "s"  # @param {type:"string"}

# Default placeholder texts
DEFAULT_ADDRESS = "Street address, apartment number, or rural route number"
DEFAULT_CITY = "City or town, state or province, and country. Include ZIP code or postal code where appropriate."

# Check if values are set (not the default placeholders)
address_filled = Street_Address and Street_Address != DEFAULT_ADDRESS
city_filled = City_Town_County_ZIP_Code and City_Town_County_ZIP_Code != DEFAULT_CITY

if address_filled and city_filled:
    print("\n✅ All address fields are filled!")
else:
    print("\n⚠️ Please fill in the following:")
    if not address_filled:
        print("  • Street address")
    if not city_filled:
        print("  • City/Town/County/ZIP")


✅ All address fields are filled!


In [7]:
!curl -L -o Schedule_C.pdf \
https://raw.githubusercontent.com/Mihailby/Taxes/main/Schedule_C.pdf

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  145k  100  145k    0     0   328k      0 --:--:-- --:--:-- --:--:--  328k


In [8]:
def fill_pdf_form_fields(input_path, output_path, values):
    reader = PdfReader(input_path)
    writer = PdfWriter()

    for page in reader.pages:
        writer.add_page(page)

    if reader.get_fields():
        writer.update_page_form_field_values(writer.pages[0], values)
    else:
        print("⚠️ No form fields detected in PDF")

    with open(output_path, "wb") as f:
        writer.write(f)

    print(f"PDF saved to: {output_path}")

In [9]:
field_values = {
    "Name": Name,
    "SSN 1": ssn1,
    "SSN 2": ssn2,
    "SSN 3": ssn3,
    "Address Where Rented": "123 Main St, NY"
}

fill_pdf_form_fields(
    "Schedule_C.pdf",
    "filled_Schedule_C.pdf",
    field_values
)

PDF saved to: filled_Schedule_C.pdf


In [10]:
from google.colab import files
files.download("filled_Schedule_C.pdf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
# Cell 1: Install and import
%pip install PyMuPDF

import fitz
import os
from google.colab import files

print(f"✅ PyMuPDF version: {fitz.version}")

✅ PyMuPDF version: ('1.27.1', '1.27.1', None)


In [17]:
# Cell 2: Download your PDF first (if needed)
!wget -O Schedule_C.pdf https://raw.githubusercontent.com/Mihailby/Taxes/main/Schedule_C.pdf

# Check if download worked
if os.path.exists('Schedule_C.pdf'):
    print(f"✅ PDF downloaded: {os.path.getsize('Schedule_C.pdf')} bytes")
else:
    print("❌ Download failed")

--2026-03-04 19:53:06--  https://raw.githubusercontent.com/Mihailby/Taxes/main/Schedule_C.pdf
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 148537 (145K) [application/octet-stream]
Saving to: ‘Schedule_C.pdf’

Schedule_C.pdf      100%[===================>] 145.06K  --.-KB/s    in 0.08s   

2026-03-04 19:53:07 (1.75 MB/s) - ‘Schedule_C.pdf’ saved [148537/148537]

✅ PDF downloaded: 148537 bytes


In [23]:
# Simplest version - just a clickable blue rectangle
def add_clickable_rectangles(input_pdf, output_pdf, links_data):
    """
    Add simple clickable blue rectangles (no symbols)
    """
    try:
        doc = fitz.open(input_pdf)
        total_icons = 0

        for link_info in links_data:
            text_to_find = link_info['text']
            url = link_info['url']
            icons_added = 0

            for page_num in range(len(doc)):
                page = doc[page_num]
                text_instances = page.search_for(text_to_find)

                for rect in text_instances:
                    # Draw a clickable blue rectangle
                    icon_rect = fitz.Rect(
                        rect.x1 + 5, rect.y0 - 2,
                        rect.x1 + 15, rect.y0 + 11
                    )

                    # Fill with blue color
                    page.draw_rect(icon_rect,
                                 color=(0, 0, 1),      # Blue border
                                 fill=(0, 0, 1),       # Solid blue fill
                                 width=0.5)

                    # Add link
                    page.insert_link({
                        "kind": fitz.LINK_URI,
                        "from": icon_rect,
                        "uri": url
                    })

                    icons_added += 1

            total_icons += icons_added
            print(f"✅ '{text_to_find}': Added {icons_added} clickable blue rectangles")

        doc.save(output_pdf)
        doc.close()
        print(f"\n✅ Complete! Added {total_icons} clickable areas to {output_pdf}")
        return True

    except Exception as e:
        print(f"❌ Error: {e}")
        return False

In [30]:
# Cell 4: Download the result
if os.path.exists('linked_schedule_c.pdf'):
    files.download('linked_schedule_c.pdf')
    print("✅ Download started!")
else:
    print("❌ Output file not found")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started!


In [31]:
# Add these improvements to your function

def add_help_link_icons_enhanced(
    input_pdf: str,
    output_pdf: str,
    text_to_find: str,
    url: str,
    *,
    icon_mode: str = "text",
    icon_text: str = "ⓘ",
    icon_png_path: str | None = None,
    icon_offset: tuple[float, float] = (6, 0),
    icon_size: float = 11,
    draw_box: bool = True,
    box_fill=(0.85, 0.92, 1.0),
    box_stroke=(0.0, 0.35, 0.85),
    text_color=(0.0, 0.35, 0.85),
    add_tooltip: str | None = "Open help link",
    box_rounded: bool = False,  # NEW: option for rounded corners
    highlight_text: bool = False,  # NEW: highlight the found text
):
    """Enhanced version with additional options"""

    doc = fitz.open(input_pdf)
    total_added = 0

    # Optional: Register a font that supports more symbols
    # if icon_mode == "text" and icon_text not in "ⓘ↗?i":
    #     # You could load a custom font here
    #     pass

    for page_index in range(len(doc)):
        page = doc[page_index]
        instances = page.search_for(text_to_find)

        if not instances:
            continue

        # NEW: Optionally highlight the found text
        if highlight_text:
            for rect in instances:
                page.draw_rect(rect, color=(1, 1, 0), fill=(1, 1, 0.8), width=0.5, overlay=True)

        for rect in instances:
            x0 = rect.x1 + icon_offset[0]
            y0 = rect.y0 + icon_offset[1]
            icon_rect = fitz.Rect(x0, y0, x0 + icon_size, y0 + icon_size)

            # Add link annotation
            page.insert_link({
                "kind": fitz.LINK_URI,
                "from": icon_rect,
                "uri": url,
            })

            # NEW: Rounded rectangle option
            if draw_box:
                if box_rounded:
                    # Draw a rectangle with rounded corners
                    radius = icon_size * 0.2
                    path = page.new_path()
                    path.draw_rect(icon_rect, radius=radius)  # Some versions support radius
                    page.draw_path(path, color=box_stroke, fill=box_fill, width=0.7)
                else:
                    page.draw_rect(icon_rect, color=box_stroke, fill=box_fill, width=0.7)

            # Draw icon
            if icon_mode == "text":
                # Center the text better
                text_width = len(icon_text) * (icon_size * 0.4)  # Approximate
                text_x = icon_rect.x0 + (icon_size - text_width) / 2
                text_y = icon_rect.y0 + icon_size * 0.75

                page.insert_text(
                    (text_x, text_y),
                    icon_text,
                    fontsize=icon_size - 2,
                    color=text_color,
                )
            else:
                # Insert image with proper scaling
                if icon_png_path:
                    page.insert_image(icon_rect, filename=icon_png_path, keep_proportion=True)

            total_added += 1

    doc.save(output_pdf, garbage=4, deflate=True)
    doc.close()

    # Summary
    print(f"\n{'='*50}")
    print(f"✅ Complete! Added {total_added} icon(s)")
    print(f"📄 Input:  {input_pdf}")
    print(f"📄 Output: {output_pdf}")
    print(f"{'='*50}")

    return total_added

In [46]:
# Example 3: Question mark for help
add_help_link_icons(
    input_pdf="Schedule_C.pdf",
    output_pdf="output_with_help.pdf",
    text_to_find="Schedule C: Deductions",
    url="https://ampl.com",
    icon_text="?",
    draw_box=True,
    box_fill=(1, 0.95, 0.85),
    box_stroke=(0.8, 0.5, 0.1),
    text_color=(0.8, 0.3, 0),
)

✅ Page 1: added 1 help icon(s) after 'Schedule C: Deductions'

✅ Done. Total icons added: 1
✅ Saved to: output_with_help.pdf


In [48]:
# Cell 5: One-liner to download all PDFs
[files.download(f) for f in os.listdir() if f.endswith('.pdf')]
print("✅ All PDF files downloaded")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ All PDF files downloaded
